# Stained-Glass Crisp

**Author:** Iman Mossavat · Fontys ICT

## What creates stained-glass crisp?

This image is made from Voronoi cells. Points are first relaxed for even spacing, then each pixel is assigned to its nearest seed, colored by a recipe, and finally streaked, cracked, and bordered with rounded lead lines.

![Final stained-glass artwork](../assets/stained_glass_crisp_preview.png)


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from math_vis import seeds_relaxed, cell_colors, glass_crisp, make_cmap, finish, save_with_meta

SEED = 4
W, H = 1300, 900
N_CELLS = 170
LLOYD = 2
rng = np.random.default_rng(SEED)

JEWEL = make_cmap([
    "#0b1d3a", "#1f4e8c", "#2aa5c9", "#58d3c0", "#f4e285",
    "#f59e3b", "#e0453a", "#8e2a6a", "#3a1d6e"
])


## Relaxing the seeds

`seeds_relaxed` scatters points randomly, then moves each point toward the center of its Voronoi cell. This produces a more even, organic spacing than pure random placement.

In [ ]:
# 1. Generate and relax seeds
seeds = seeds_relaxed(N_CELLS, W, H, rng, iters=LLOYD)
fig, ax = plt.subplots(figsize=(10, 6.5))
ax.scatter(seeds[:,0], seeds[:,1], s=8, c='#f4e285')
ax.set_xlim(0, W); ax.set_ylim(H, 0); ax.set_aspect('equal'); ax.axis('off')
ax.set_title('Lloyd-relaxed Voronoi seeds')
plt.tight_layout(); plt.show()


> **Question:** What would the cells look like if you set `LLOYD=0` and skipped relaxation? Predict, then try it.

<details><summary>Answer</summary>

DRAFT: author to review

With `LLOYD=0` the seeds stay at their initial random positions. The Voronoi cells keep the highly irregular sizes and shapes of a random point set: some cells are tiny, others are large and elongated, and there is no even, organic spacing. Running Lloyd relaxation moves each seed toward its cell centroid and makes the distribution much more regular.

</details>

## Coloring the cells

`cell_colors` assigns a scalar value to each seed based on position and a little randomness. That scalar is then mapped through a palette to give every cell its color.

In [ ]:
# 2. Assign colors to cells
t = cell_colors(seeds, W, H, rng, trend=0.5, smooth=20, jitter=0.07)

# visualize just the cell assignment
from scipy.spatial import cKDTree
X, Y = np.meshgrid(np.arange(W)+0.5, np.arange(H)+0.5)
P = np.column_stack([X.ravel(), Y.ravel()])
cell = cKDTree(seeds).query(P)[1]
flat = (JEWEL(t)[cell, :3] * 255).reshape(H, W, 3).astype(np.uint8)

fig, ax = plt.subplots(figsize=(10, 6.5))
ax.imshow(flat, origin='lower'); ax.axis('off')
ax.set_title('Flat cells from color recipe')
plt.tight_layout(); plt.show()


> **Question:** The color recipe mixes position (`trend`) with randomness. Which do you think has more influence in the current image?

<details><summary>Answer</summary>

DRAFT: author to review

`cell_colors` computes `trend * diag + (1-trend) * squash(noise) + jitter`. With `trend=0.5` both terms are weighted equally, so neither dominates. The diagonal position term (`(x/W + y/H)/2`) creates large-scale colour gradients across the image, while the smoothed noise and jitter add local variation. You can see both: broad coherent regions with smaller random shifts inside them.

</details>

## Building the image stage by stage

`glass_crisp` adds warp, lead lines, rounded edges, streaks, cracks, and contours. Each stage is a small visual effect; together they create the illusion of old glass.

In [ ]:
# 3. Build the image stage by stage
img, stages = glass_crisp(
    seeds, t, JEWEL, W, H, rng=rng,
    warp=16,
    lead_px=1.8,
    round_px=4.0,
    streak=0.035,
    streak_light=0.10,
    crack_alpha=0.30,
    contour_alpha=0.28,
    stages=True
)

stage_names = ['flat', 'streaks', 'cracks', 'leaded', 'final']
fig, axes = plt.subplots(2, 3, figsize=(15, 8.6))
for ax, name in zip(axes.ravel(), stage_names):
    if name == 'final':
        # close-up of the final image in the last panel
        crop = stages['final'][H // 5: H // 5 + 300, W // 4: W // 4 + 450]
        ax.imshow(crop)
        ax.set_title('final (close-up)')
    else:
        ax.imshow(stages[name])
        ax.set_title(name)
    ax.axis('off')
# hide any unused panels in the 2x3 grid
for ax in axes.ravel()[len(stage_names):]:
    ax.axis('off')
plt.tight_layout(); plt.show()


> **Question:** Which stage makes the flat cells look three-dimensional — the streaks, the rounded lead, or the cracks?

<details><summary>Answer</summary>

DRAFT: author to review

The **streaks** stage is the main one. `apply_streaks` adds soft parallel bands inside each cell and a weak lens tilt (`tilt=0.06`) that brightens and darkens parts of the cell based on its orientation. That directional shading is what makes the flat cells look three-dimensional. The rounded lead and cracks add material detail, but they mainly affect the lead and surface texture, not the cell volume.

</details>

## Experiments

Try changing `N_CELLS`, `LLOYD`, `warp`, `streak`, `crack_alpha`, or the palette. Start with one change at a time and re-run from the parameter cell onward.

In [ ]:
# 4. Final render: finishing, save with metadata, and preview
final = finish(stages['final'], grain=2, vignette=0.0, display=False)

from IPython.display import display
from PIL import Image

preview = Image.fromarray(final).resize((800, 554), Image.Resampling.LANCZOS)
display(preview)

_ = save_with_meta(
    final,
    'stained_glass_crisp.png',
    source_notebook='notebooks/demo_stained_glass_crisp.ipynb',
    seed=SEED,
    render_width=W,
    render_height=H,
    n_cells=N_CELLS,
    lloyd_iters=LLOYD,
    palette='JEWEL',
    glass_crisp_params={
        'warp': 16,
        'lead_px': 1.8,
        'round_px': 4.0,
        'streak': 0.035,
        'streak_light': 0.10,
        'crack_alpha': 0.30,
        'contour_alpha': 0.28,
    },
    finish_params={'grain': 2},
    display_resize={'width': 800, 'height': 554, 'filter': 'LANCZOS'},
)


## What did we learn?

- **Lloyd relaxation** makes Voronoi cells more regular without making them perfectly uniform.
- A **color recipe** based on cell position creates coherent regions.
- Small details (streaks, cracks, rounded lead) add material realism.

**Open questions:**
1. What happens if you skip Lloyd relaxation entirely?
2. How does the `warp` parameter change the geometry of the cells?

**Further experiment:** Try a palette with only warm colors, or increase `crack_alpha`.

## Further reading

- Conceptual background: [Geometry from a collection of points](conceptualGuide.md#3-geometry-from-a-collection-of-points) in `conceptualGuide.md`.
- Implementation details: [Stained glass (crisp)](recipes.md#6-stained-glass-crisp) in `recipes.md`.